# Лабораторная работа: Автоматизированное машинное обучение - AutoML

## Цель работы:
1. Освоить основные принципы автоматизированного машинного обучения (AutoML).
2. Научиться использовать популярные AutoML-библиотеки, такие как **AutoKeras** и **H2O**.
3. Сравнить производительность различных моделей машинного обучения, включая глубокие нейронные сети, используя leaderboard в H2O.
4. Получить практические навыки анализа данных и интерпретации результатов.

---

## Задача:
Рассмотрим задачу прогнозирования состояния промышленного оборудования на основе данных с датчиков. Данные содержат информацию о температуре, давлении, вибрации и других параметрах. Задача — предсказать, будет ли оборудование работать нормально или выйдет из строя в ближайшее время (классификация).

---

## Подготовка к работе:

### Необходимое программное обеспечение:
- Python 3.x
- Библиотеки: `pandas`, `numpy`, `scikit-learn`, `autokeras`, `h2o`
- IDE: Jupyter Notebook или PyCharm

### Установка библиотек:
```bash
pip install pandas numpy scikit-learn autokeras h2o
```

### Исходные данные:
Для работы будем использовать набор данных [SECOM](https://archive.ics.uci.edu/ml/datasets/SECOM), который содержит данные с датчиков промышленного оборудования. Данные можно загрузить по ссылке: [SECOM Dataset](https://archive.ics.uci.edu/ml/machine-learning-databases/secom/).

---

## Часть 1: Работа с AutoKeras

### Шаг 1. Загрузка и предобработка данных
```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Загрузка данных
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom.data"
data = pd.read_csv(url, header=None, sep=" ")

# Загрузка меток
labels_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom_labels.data"
labels = pd.read_csv(labels_url, header=None, sep=" ", usecols=[0])

# Обработка пропущенных значений
data = data.fillna(data.mean())

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)

# Масштабирование данных
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
```

### Шаг 2. Создание модели с использованием AutoKeras
```python
import autokeras as ak

# Создание классификатора AutoKeras
clf = ak.StructuredDataClassifier(max_trials=10, overwrite=True)

# Обучение модели
clf.fit(X_train, y_train.values.ravel(), epochs=50)

# Оценка модели
accuracy = clf.evaluate(X_test, y_test.values.ravel())
print(f"Accuracy: {accuracy}")
```

### Пояснения:
- `StructuredDataClassifier` автоматически подбирает архитектуру модели, оптимизатор и гиперпараметры.
- `max_trials=10` задает максимальное количество попыток для поиска лучшей модели.
- Выводится точность модели на тестовой выборке.

---

## Часть 2: Работа с H2O

### Шаг 1. Инициализация H2O
```python
import h2o
from h2o.automl import H2OAutoML

# Инициализация H2O
h2o.init()

# Преобразование данных в формат H2O
train_h2o = h2o.H2OFrame(pd.concat([pd.DataFrame(X_train), pd.DataFrame(y_train)], axis=1))
test_h2o = h2o.H2OFrame(pd.concat([pd.DataFrame(X_test), pd.DataFrame(y_test)], axis=1))

# Разделение на признаки и целевую переменную
x = train_h2o.columns[:-1]
y = train_h2o.columns[-1]
train_h2o[y] = train_h2o[y].asfactor()  # Преобразование целевой переменной в категориальный тип
test_h2o[y] = test_h2o[y].asfactor()
```

### Шаг 2. Обучение моделей с помощью H2O AutoML
```python
# Создание и обучение AutoML
aml = H2OAutoML(max_models=10, seed=42, include_algos=["DeepLearning", "GBM", "GLM", "DRF"])
aml.train(x=x, y=y, training_frame=train_h2o)

# Вывод leaderboard
lb = aml.leaderboard
print(lb)
```

### Шаг 3. Оценка лучшей модели
```python
# Получение лучшей модели
best_model = aml.leader

# Оценка на тестовой выборке
performance = best_model.model_performance(test_data=test_h2o)
print(performance)
```

### Пояснения:
- `H2OAutoML` автоматически обучает несколько моделей, включая глубокие нейронные сети (`DeepLearning`), градиентный бустинг (`GBM`), обобщенные линейные модели (`GLM`) и случайные леса (`DRF`).
- `leaderboard` показывает ранжирование моделей по их производительности.
- Лучшая модель выбирается автоматически и может быть использована для дальнейшего анализа.

---

## Часть 3: Анализ результатов

### Сравнение моделей
1. Сравните точность моделей, полученных с помощью AutoKeras и H2O.
2. Проанализируйте leaderboard в H2O. Какие модели показали наилучшие результаты? Почему?
3. Объясните, как глубокие нейронные сети повлияли на качество решения задачи.

### Интерпретация результатов
1. Какие признаки оказались наиболее важными для прогнозирования состояния оборудования?
2. Можно ли улучшить результаты, добавив дополнительные данные или изменив параметры обучения?

---

## Дополнительные задания:
1. Попробуйте использовать другие AutoML-библиотеки, такие как TPOT или MLJAR.
2. Измените задачу с классификации на регрессию (например, предсказание времени до выхода из строя оборудования).
3. Исследуйте влияние различных гиперпараметров на производительность моделей.

---

## Заключение:
В ходе лабораторной работы вы освоили использование AutoML-библиотек для решения задач управления в технических системах. Вы научились автоматически подбирать модели, анализировать их производительность и интерпретировать результаты. Эти навыки будут полезны для решения реальных задач в промышленности. 

**Ответ:** {Лабораторная работа успешно подготовлена и включает примеры с AutoKeras, H2O и leaderboard для сравнения моделей.}